In [ ]:
# Writing full training code for GA-BiLSTM model with CNN feature extraction into a single file
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
from PIL import Image

# ---------------------------
# Configuration
# ---------------------------
DATASET_PATH = "./dataset"  # contains "stroke/" and "normal/" folders
BATCH_SIZE = 16
NUM_EPOCHS = 10
INPUT_FEATURES = 256
NUM_CLASSES = 2
KFOLDS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# BiLSTM Model
# ---------------------------
class BiLSTMNet(nn.Module):
    def __init__(self, input_dim=INPUT_FEATURES, hidden_dim=64):
        super(BiLSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, NUM_CLASSES)

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, seq_len=1, input_dim)
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out

# ---------------------------
# Feature Extractor (AlexNet)
# ---------------------------
alexnet = models.alexnet(pretrained=True)
alexnet.classifier = nn.Sequential(*list(alexnet.classifier.children())[:-1])
alexnet.eval()
alexnet.to(DEVICE)

transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor()
])

# ---------------------------
# Custom Dataset with Features
# ---------------------------
class FeatureDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(path).convert("RGB")
        image = transform(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            features = alexnet(image).squeeze().cpu().numpy()
        selected_features = features[:INPUT_FEATURES]  # Simulated GA-based selection
        return torch.tensor(selected_features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# ---------------------------
# Load image paths and labels
# ---------------------------
def load_data_paths(dataset_path):
    image_paths, labels = [], []
    for label, category in enumerate(["Normal", "Stroke"]):
        category_path = os.path.join(dataset_path, category)
        for fname in os.listdir(category_path):
            if fname.endswith((".jpg", ".png", ".jpeg")):
                image_paths.append(os.path.join(category_path, fname))
                labels.append(label)
    return image_paths, labels

# ---------------------------
# Training and Evaluation
# ---------------------------
def train_model():
    image_paths, labels = load_data_paths(DATASET_PATH)
    image_paths, labels = np.array(image_paths), np.array(labels)

    kfold = KFold(n_splits=KFOLDS, shuffle=True, random_state=42)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(image_paths)):
        print(f"\\n--- Fold {fold+1} ---")
        train_set = FeatureDataset(image_paths[train_idx], labels[train_idx])
        val_set = FeatureDataset(image_paths[val_idx], labels[val_idx])

        train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

        model = BiLSTMNet().to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        # Train
        for epoch in range(NUM_EPOCHS):
            model.train()
            for X, y in train_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                out = model(X)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {loss.item():.4f}")

        # Evaluate
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(DEVICE)
                out = model(X)
                preds = torch.argmax(out, dim=1).cpu().numpy()
                y_true.extend(y.numpy())
                y_pred.extend(preds)

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        print(f"Fold {fold+1} - Acc: {acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f}")
        fold_metrics.append((acc, prec, rec, f1))

    # Save the final trained model
    torch.save(model.state_dict(), "bilstm_model.pth")
    print("\\nModel saved as bilstm_model.pth")

    # Summary
    avg = np.mean(fold_metrics, axis=0)
    print(f"\\nAverage Performance across {KFOLDS} folds:")
    print(f"Accuracy: {avg[0]:.4f}, Precision: {avg[1]:.4f}, Recall: {avg[2]:.4f}, F1: {avg[3]:.4f}")

if __name__ == "__main__":
    train_model()

with open("/mnt/data/train_stroke_model.py", "w") as f:
    f.write(train_code)

"/mnt/data/train_stroke_model.py"


ModuleNotFoundError: No module named 'torch'

In [6]:
import os
from PIL import Image
import numpy as np

dataset_dir = './dataset'
classes = ['Normal', 'Stroke']

summary = {}

for class_name in classes:
    class_path = os.path.join(dataset_dir, class_name)
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_shapes = []

    for img_name in image_files:
        img_path = os.path.join(class_path, img_name)
        try:
            with Image.open(img_path) as img:
                image_shapes.append(img.size)  # (width, height)
        except Exception as e:
            print(f"Error loading {img_name}: {e}")

    image_shapes = np.array(image_shapes)
    unique_shapes, counts = np.unique(image_shapes, axis=0, return_counts=True)

    summary[class_name] = {
        'total_images': len(image_files),
        'unique_shapes': list(zip(unique_shapes.tolist(), counts.tolist()))
    }

# Display summary
for cls, stats in summary.items():
    print(f"\n📂 {cls}")
    print(f"Total images: {stats['total_images']}")
    print("Image dimensions (width x height) and their counts:")
    for shape, count in stats['unique_shapes']:
        print(f"  - {shape[0]}x{shape[1]}: {count} images")



📂 Normal
Total images: 1551
Image dimensions (width x height) and their counts:
  - 650x650: 1551 images

📂 Stroke
Total images: 950
Image dimensions (width x height) and their counts:
  - 650x650: 950 images


In [ ]:
# Creating a new complete training script with enhanced preprocessing and a more accurate setup
# for GA_BiLSTM stroke classification to aim for higher accuracy (closer to 96.5%)
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from PIL import Image
from tqdm import tqdm
import random


# Configuration
DATASET_DIR = "./dataset"
BATCH_SIZE = 16
EPOCHS = 20
INPUT_FEATURES = 256
KFOLD = 5
LEARNING_RATE = 0.0005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# BiLSTM Model
class BiLSTMNet(nn.Module):
    def __init__(self, input_dim=INPUT_FEATURES, hidden_dim=64):
        super(BiLSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):
        x = x.unsqueeze(1)  # Shape: (batch, seq_len=1, input_dim)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# Feature Extractor using AlexNet
alexnet = models.alexnet(pretrained=True)
alexnet.classifier = nn.Sequential(*list(alexnet.classifier.children())[:-1])
alexnet.to(DEVICE)
alexnet.eval()

# Image transformation
transform = transforms.Compose([
    transforms.Resize((227, 227)),
    transforms.ToTensor(),
])

# Custom Dataset
class StrokeDataset(Dataset):
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            features = alexnet(image).squeeze().cpu().numpy()

        selected_features = features[:INPUT_FEATURES]  # Simulated GA selection
        return torch.tensor(selected_features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# Load dataset paths and labels
def load_dataset():
    normal_dir = os.path.join(DATASET_DIR, 'Normal')
    stroke_dir = os.path.join(DATASET_DIR, 'Stroke')

    normal_images = [os.path.join(normal_dir, f) for f in os.listdir(normal_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    stroke_images = [os.path.join(stroke_dir, f) for f in os.listdir(stroke_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Balance dataset by selecting only 950 Normal images
    random.seed(42)
    normal_images = random.sample(normal_images, 950)
    stroke_images = stroke_images[:950]  # already 950

    image_paths = normal_images + stroke_images
    labels = [0] * 950 + [1] * 950  # 0: Normal, 1: Stroke

    # Shuffle the dataset
    combined = list(zip(image_paths, labels))
    random.shuffle(combined)
    image_paths[:], labels[:] = zip(*combined)

    return np.array(image_paths), np.array(labels)


# Training function
def train_model():
    image_paths, labels = load_dataset()
    skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=42)

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
        print(f"\\n--- Fold {fold + 1} ---")

        train_data = StrokeDataset(image_paths[train_idx], labels[train_idx])
        val_data = StrokeDataset(image_paths[val_idx], labels[val_idx])

        train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

        model = BiLSTMNet().to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            for X, y in train_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(X)
                loss = criterion(outputs, y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f}")

        # Evaluation
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(DEVICE)
                outputs = model(X)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                y_true.extend(y.numpy())
                y_pred.extend(preds)

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        print(f"Fold {fold+1} - Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
        fold_results.append((acc, prec, rec, f1))

    # Average results
    fold_results = np.array(fold_results)
    print("\\n=== Final Average Scores ===")
    print(f"Accuracy: {fold_results[:,0].mean():.4f}")
    print(f"Precision: {fold_results[:,1].mean():.4f}")
    print(f"Recall: {fold_results[:,2].mean():.4f}")
    print(f"F1 Score: {fold_results[:,3].mean():.4f}")

    # Save the model
    torch.save(model.state_dict(), "bilstm_model.pth")
    print("Trained model saved to bilstm_model.pth")

if __name__ == "__main__":
    train_model()

# # Save to file
# file_path = "/mnt/data/train_stroke_model_final.py"
# with open(file_path, "w") as f:
#     f.write(enhanced_code)

# file_path


/home/k8sdev1/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/k8sdev1/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


\n--- Fold 1 ---
Epoch 1/20 - Loss: 0.6877
Epoch 2/20 - Loss: 0.6663
Epoch 3/20 - Loss: 0.6429
